In [5]:
!pip install -q langchain

In [6]:
!pip install -q langchain-community

In [8]:
!pip install -q sentence-transformers

In [7]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 14.8 MB/s eta 0:00:00


In [9]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.2 MB/s eta 0:00:00


In [10]:
!pip install -q transformers

In [11]:
!pip install -q accelerate

In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

pdf_folder = "/content/data"

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        path = os.path.join(pdf_folder, file)

        loader = PyPDFLoader(path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source_file"] = file

        all_docs.extend(docs)

print("Total pages loaded:", len(all_docs))

Total pages loaded: 542


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))

Total chunks: 1646


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2170/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector DB created")

Vector DB created


In [18]:
vectorstore.save_local(
    "pharmareg_faiss"
)

In [19]:
query = "How should vaccines be stored?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print("\n")
    print("="*80)
    print(f"Result {i+1}")
    print(doc.metadata)
    print(doc.page_content[:1000])



Result 1
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2020-05-12T15:55:51+02:00', 'moddate': '2020-05-12T15:56:00+02:00', 'source': '/content/data/trs1025-annex7-(1).pdf', 'total_pages': 32, 'page': 30, 'page_label': '31', 'source_file': 'trs1025-annex7-(1).pdf'}
medicinedocs/documents/s21492en/s21492en.pdf, accessed 5 December 2019).
 ■ Good manufacturing practices: guidelines on validation. In: WHO Expert Committee on 
Specifications for Pharmaceutical Preparations: fifty-third report. Geneva: World Health 
Organization; 2019: Annex 3 (WHO Technical Report Series, No. 1019; https://www.who.int/
medicines/areas/quality_safety/quality_assurance/WHO_TRS_1019_Annex3.pdf?ua=1, accessed 
29 January 2020). 
 ■ Guidelines on the international packaging and shipping of vaccines. Geneva: World Health 
Organization; 2019 (https://www.who.int/immunization_standards/vaccine_quality/Vaccines_
Shipping_Guidelines082019.pdf?ua=1, accessed 5 D

In [20]:
queries = [
    "What is pharmacovigilance?",
    "How should biological products be transported?",
    "What are hospital pharmacy requirements?",
    "What is a drug recall?",
    "What information must appear on labels?"
]

for q in queries:
    results = vectorstore.similarity_search(q, k=1)

    print("\nQUESTION:", q)
    print(results[0].page_content[:500])


QUESTION: What is pharmacovigilance?
Guidelines on National Pharmacovigilance System (Edition 04)  
 
Page 90 of 104 
Division of Pharmacy Services                                                                          Effective Date: 10-10-2025 
 
11.2.2 Process indicators: These indicators assess the extent of pharmacovigilance activities. They 
focus on the constellation of activities which describe the mechanism of pharmacovigilance – the 
collection, collation, analysis and evaluation of ADR reports. They also consider other

QUESTION: How should biological products be transported?
Good Distribution Practices for Pharmaceutical and Biological Drug Products (Edition 01)  
Page 32 of 35 
Division of Quality Assurance and Lab Testing, DRAP Effective Date: 01-03-2023 
that may compromise their quality and integrity. A risk -based approach should be 
utilized when planning transportation. 
15.2. Transportation 
The required storage conditions for drug products should be maintained d

In [21]:
query = "What is pharmacovigilance?"

results = vectorstore.similarity_search_with_score(
    query,
    k=5
)

for i, (doc, score) in enumerate(results):

    print("\n" + "="*100)
    print(f"RESULT {i+1}")
    print("="*100)

    print("Document:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print("Score:", score)

    print("\nText:")
    print(doc.page_content[:1200])


RESULT 1
Document: NPVG-guidelines-reviewed-final-10-10-24-1.pdf
Page: 89
Score: 0.4307651

Text:
Guidelines on National Pharmacovigilance System (Edition 04)  
 
Page 90 of 104 
Division of Pharmacy Services                                                                          Effective Date: 10-10-2025 
 
11.2.2 Process indicators: These indicators assess the extent of pharmacovigilance activities. They 
focus on the constellation of activities which describe the mechanism of pharmacovigilance – the 
collection, collation, analysis and evaluation of ADR reports. They also consider other activities 
which influence those listed above. These are measures that assess directly or indirectly the extent 
to which the system is operating. 
11.2.3 Outcome or impact indicators:  These indicators measure the effects (results and changes) 
of pharmacovigilance activities. They measure the extent of realization of the pharmacovigilance 
objectives which, in essence, constitute ensuring patie

In [22]:
def retrieve_documents(query, k=5):

    results = vectorstore.similarity_search_with_score(
        query,
        k=k
    )

    retrieved = []

    for doc, score in results:

        retrieved.append({
            "text": doc.page_content,
            "source": doc.metadata.get("source_file"),
            "page": doc.metadata.get("page"),
            "score": float(score)
        })

    return retrieved

In [23]:
results = retrieve_documents(
    "How should biological products be transported?"
)

for result in results:
    print(result)

{'text': 'Good Distribution Practices for Pharmaceutical and Biological Drug Products (Edition 01)  \nPage 32 of 35 \nDivision of Quality Assurance and Lab Testing, DRAP Effective Date: 01-03-2023 \nthat may compromise their quality and integrity. A risk -based approach should be \nutilized when planning transportation. \n15.2. Transportation \nThe required storage conditions for drug products should be maintained during  \ntransportation within the defined limits as  described on the outer packaging and/or  \nrelevant packaging information . The distributor(s) must ensure that drug products are \nnot subjected to prolonged periods of storage during transportation. \nWhere contract service providers are used, the distributor/wholesaler must make itself \naware of the operating procedures of that party (e.g., by audit). This assessment should \ninclude examination of the transportation methods and routes. The \ndistributor/wholesaler should be fully aware of, and agree to , any operatio

In [24]:
evaluation_questions = [
    "What is pharmacovigilance?",
    "How should biological products be transported?",
    "What are the minimum requirements for hospital pharmacies?",
    "What is a drug recall?",
    "What information should be included on pharmaceutical labels?",
    "What are the requirements for medicine storage?",
    "How should temperature-sensitive pharmaceutical products be handled?",
    "What are the responsibilities of pharmaceutical distributors?",
    "What is an adverse drug reaction?",
    "What are the requirements for monitoring medicine storage temperature?",
    "What are good distribution practices?",
    "What are the requirements for transportation of pharmaceutical products?",
    "What is the purpose of pharmacovigilance?",
    "What procedures should be followed during a drug recall?",
    "How should medicines requiring light protection be stored?",
    "What are the responsibilities of a hospital pharmacy?",
    "What are the requirements for maintaining pharmacy records?",
    "How should defective therapeutic goods be handled?",
    "What are the requirements for pharmaceutical product quality during transportation?",
    "What is the role of DRAP in pharmaceutical regulation?"
]

In [25]:
for i, question in enumerate(evaluation_questions, 1):

    results = vectorstore.similarity_search_with_score(
        question,
        k=3
    )

    print(f"\n{i}. {question}")

    for j, (doc, score) in enumerate(results, 1):

        print(
            f"   Result {j}: "
            f"{doc.metadata.get('source_file')} "
            f"| Page {doc.metadata.get('page')} "
            f"| Score {score:.4f}"
        )


1. What is pharmacovigilance?
   Result 1: NPVG-guidelines-reviewed-final-10-10-24-1.pdf | Page 89 | Score 0.4308
   Result 2: NPVG-guidelines-reviewed-final-10-10-24-1.pdf | Page 88 | Score 0.4378
   Result 3: NPVG-guidelines-reviewed-final-10-10-24-1.pdf | Page 101 | Score 0.5115

2. How should biological products be transported?
   Result 1: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf | Page 31 | Score 0.8444
   Result 2: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf | Page 31 | Score 0.9140
   Result 3: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf | Page 32 | Score 0.9242

3. What are the minimum requirements for hospital pharmacies?
   Result 1: PHSR-GL-HP-012-Guidelines-on-Standards-for-Establishment-of-Hospital-Pharmacies-in-Pakistan-1.pdf | Page 13 | Score 0.5842
   Result 2: PHSR-GL-HP-012-Guidelines-on-Standards-for-Establishment-of-Hospital-Pharmacies-in-Pakistan-1.pdf | Page 25 | Score 0

In [26]:
!pip install -q openai

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [29]:
def create_rag_prompt(question, retrieved_docs):

    context = ""

    for i, doc in enumerate(retrieved_docs, 1):
        context += f"""
SOURCE {i}
Document: {doc["source"]}
Page: {doc["page"]}

Content:
{doc["text"]}

-----------------------------------
"""

    prompt = f"""
You are PharmaRegAI, a pharmaceutical regulatory
information assistant.

Answer the question using ONLY the regulatory
documents provided below.

Rules:
1. Do not invent information.
2. Do not use outside knowledge.
3. If the documents do not contain enough information,
   say that the available documents do not provide
   sufficient information.
4. Give a clear and concise answer.
5. Mention the document name and page number when possible.

REGULATORY DOCUMENTS:

{context}

QUESTION:

{question}

ANSWER:
"""

    return prompt

In [30]:
def ask_pharmareg(question, k=5):

    retrieved = retrieve_documents(question, k=k)

    prompt = create_rag_prompt(
        question,
        retrieved
    )

    result = generator(prompt)

    answer = result[0]["generated_text"]

    return answer, retrieved

In [33]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

print("Generation function ready!")

Generation function ready!


In [34]:
def ask_pharmareg(question, k=5):

    retrieved = retrieve_documents(question, k=k)

    prompt = create_rag_prompt(
        question,
        retrieved
    )

    answer = generate_answer(prompt)

    return answer, retrieved

print("PharmaRegAI RAG function ready!")

PharmaRegAI RAG function ready!


In [35]:
question = "What is pharmacovigilance?"

answer, sources = ask_pharmareg(question)

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(question)

print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(answer)

QUESTION
What is pharmacovigilance?

ANSWER
ensuring patient safety


In [36]:
print("\n" + "=" * 80)
print("SOURCES")
print("=" * 80)

for i, source in enumerate(sources, 1):
    print(f"\n{i}. {source['source']}")
    print(f"   Page: {source['page']}")
    print(f"   Score: {source['score']:.4f}")


SOURCES

1. NPVG-guidelines-reviewed-final-10-10-24-1.pdf
   Page: 89
   Score: 0.4308

2. NPVG-guidelines-reviewed-final-10-10-24-1.pdf
   Page: 88
   Score: 0.4378

3. NPVG-guidelines-reviewed-final-10-10-24-1.pdf
   Page: 101
   Score: 0.5115

4. NPVG-guidelines-reviewed-final-10-10-24-1.pdf
   Page: 100
   Score: 0.5152

5. NPVG-guidelines-reviewed-final-10-10-24-1.pdf
   Page: 98
   Score: 0.5175


In [37]:
evaluation_results = []

for i, question in enumerate(evaluation_questions, 1):

    print(f"Processing {i}/20: {question}")

    answer, sources = ask_pharmareg(question)

    evaluation_results.append({
        "question": question,
        "answer": answer,
        "sources": sources
    })

print("\nEvaluation completed!")

Processing 1/20: What is pharmacovigilance?
Processing 2/20: How should biological products be transported?
Processing 3/20: What are the minimum requirements for hospital pharmacies?
Processing 4/20: What is a drug recall?
Processing 5/20: What information should be included on pharmaceutical labels?
Processing 6/20: What are the requirements for medicine storage?
Processing 7/20: How should temperature-sensitive pharmaceutical products be handled?
Processing 8/20: What are the responsibilities of pharmaceutical distributors?
Processing 9/20: What is an adverse drug reaction?
Processing 10/20: What are the requirements for monitoring medicine storage temperature?
Processing 11/20: What are good distribution practices?
Processing 12/20: What are the requirements for transportation of pharmaceutical products?
Processing 13/20: What is the purpose of pharmacovigilance?
Processing 14/20: What procedures should be followed during a drug recall?
Processing 15/20: How should medicines requir

In [38]:
import pandas as pd

evaluation_table = []

for result in evaluation_results:

    top_source = result["sources"][0]

    evaluation_table.append({
        "Question": result["question"],
        "Answer": result["answer"],
        "Top Source": top_source["source"],
        "Page": top_source["page"],
        "Retrieval Score": top_source["score"]
    })

evaluation_df = pd.DataFrame(evaluation_table)

evaluation_df

,Question,Answer,Top Source,Page,Retrieval Score
0,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765
1,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425
2,What are the minimum requirements for hospital...,"adequate office space, infrastructure, and off...",PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,13,0.584216
3,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091
4,What information should be included on pharmac...,"name of the product; active ingredient(s), typ...",NPVG-guidelines-reviewed-final-10-10-24-1.pdf,43,0.645951
5,What are the requirements for medicine storage?,Temperature and humidity of the medication sto...,p1022drap_1.pdf,26,0.674713
6,How should temperature-sensitive pharmaceutica...,Model guidance for the storage and transport o...,trs1019-annex5-who-guidelines-on-import-proced...,11,0.614655
7,What are the responsibilities of pharmaceutica...,Distributors/ wholesalers are required to repo...,Guidelines-on-Good-Distribution-Practices-for-...,5,0.564630
8,What is an adverse drug reaction?,a response to a drug or therapeutic goods whic...,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,5,0.441929
9,What are the requirements for monitoring medic...,Temperature and humidity of the medication sto...,Guidelines-on-Good-Distribution-Practices-for-...,22,0.547578


In [39]:
evaluation_labels = [
    {
        "question": "What is pharmacovigilance?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "How should biological products be transported?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the minimum requirements for hospital pharmacies?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What is a drug recall?",
        "correct": 1,
        "complete": 1,
        "source_relevant": 1
    },
    {
        "question": "What information should be included on pharmaceutical labels?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the requirements for medicine storage?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "How should temperature-sensitive pharmaceutical products be handled?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the responsibilities of pharmaceutical distributors?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What is an adverse drug reaction?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the requirements for monitoring medicine storage temperature?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are good distribution practices?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the requirements for transportation of pharmaceutical products?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What is the purpose of pharmacovigilance?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What procedures should be followed during a drug recall?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "How should medicines requiring light protection be stored?",
        "correct": 1,
        "complete": 1,
        "source_relevant": 1
    },
    {
        "question": "What are the responsibilities of a hospital pharmacy?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the requirements for maintaining pharmacy records?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "How should defective therapeutic goods be handled?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What are the requirements for pharmaceutical product quality during transportation?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    },
    {
        "question": "What is the role of DRAP in pharmaceutical regulation?",
        "correct": 1,
        "complete": 0,
        "source_relevant": 1
    }
]

In [40]:
import pandas as pd

labels_df = pd.DataFrame(evaluation_labels)

print("Baseline Evaluation")
print("=" * 50)

accuracy = labels_df["correct"].mean()
completeness = labels_df["complete"].mean()
source_relevance = labels_df["source_relevant"].mean()

print(f"Answer Correctness: {accuracy:.2%}")
print(f"Answer Completeness: {completeness:.2%}")
print(f"Source Relevance: {source_relevance:.2%}")

Baseline Evaluation
Answer Correctness: 100.00%
Answer Completeness: 10.00%
Source Relevance: 100.00%


In [43]:
import pandas as pd

evaluation_table = []

for item in evaluation_results:

    top_source = item["sources"][0]

    evaluation_table.append({
        "question": item["question"],
        "answer": item["answer"],
        "top_source": top_source["source"],
        "page": top_source["page"],
        "retrieval_distance": top_source["score"]
    })

evaluation_df = pd.DataFrame(evaluation_table)

evaluation_df.head()

,question,answer,top_source,page,retrieval_distance
0,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765
1,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425
2,What are the minimum requirements for hospital...,"adequate office space, infrastructure, and off...",PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,13,0.584216
3,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091
4,What information should be included on pharmac...,"name of the product; active ingredient(s), typ...",NPVG-guidelines-reviewed-final-10-10-24-1.pdf,43,0.645951


In [44]:
evaluation_df

,question,answer,top_source,page,retrieval_distance
0,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765
1,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425
2,What are the minimum requirements for hospital...,"adequate office space, infrastructure, and off...",PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,13,0.584216
3,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091
4,What information should be included on pharmac...,"name of the product; active ingredient(s), typ...",NPVG-guidelines-reviewed-final-10-10-24-1.pdf,43,0.645951
5,What are the requirements for medicine storage?,Temperature and humidity of the medication sto...,p1022drap_1.pdf,26,0.674713
6,How should temperature-sensitive pharmaceutica...,Model guidance for the storage and transport o...,trs1019-annex5-who-guidelines-on-import-proced...,11,0.614655
7,What are the responsibilities of pharmaceutica...,Distributors/ wholesalers are required to repo...,Guidelines-on-Good-Distribution-Practices-for-...,5,0.564630
8,What is an adverse drug reaction?,a response to a drug or therapeutic goods whic...,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,5,0.441929
9,What are the requirements for monitoring medic...,Temperature and humidity of the medication sto...,Guidelines-on-Good-Distribution-Practices-for-...,22,0.547578


In [45]:
print("Retrieval Statistics")
print("=" * 50)

print("Average retrieval distance:",
      round(evaluation_df["retrieval_distance"].mean(), 4))

print("Best retrieval distance:",
      round(evaluation_df["retrieval_distance"].min(), 4))

print("Worst retrieval distance:",
      round(evaluation_df["retrieval_distance"].max(), 4))

Retrieval Statistics
Average retrieval distance: 0.6191
Best retrieval distance: 0.4308
Worst retrieval distance: 0.8444


In [46]:
best_cases = evaluation_df.sort_values(
    "retrieval_distance"
).head(5)

best_cases

,question,answer,top_source,page,retrieval_distance
0,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765
8,What is an adverse drug reaction?,a response to a drug or therapeutic goods whic...,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,5,0.441929
18,What are the requirements for pharmaceutical p...,The required storage conditions for drug produ...,Guidelines-on-Good-Distribution-Practices-for-...,31,0.508347
9,What are the requirements for monitoring medic...,Temperature and humidity of the medication sto...,Guidelines-on-Good-Distribution-Practices-for-...,22,0.547578
15,What are the responsibilities of a hospital ph...,Care and operational responsibilities. 6.6 The...,PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,7,0.552046


In [47]:
worst_cases = evaluation_df.sort_values(
    "retrieval_distance",
    ascending=False
).head(5)

worst_cases

,question,answer,top_source,page,retrieval_distance
1,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425
10,What are good distribution practices?,Good Storage Practice (GSP) and Good Distribut...,Guidelines-on-Good-Distribution-Practices-for-...,8,0.811398
3,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091
17,How should defective therapeutic goods be hand...,"within the distribution chain, both nationally...",Guidelines-on-Good-Distribution-Practices-for-...,25,0.694534
16,What are the requirements for maintaining phar...,Records must include at least the following in...,p1022drap_1.pdf,21,0.684178


In [48]:
unsupported_question = (
    "What is DRAP's regulation for manufacturing medicines on Mars?"
)

answer, sources = ask_pharmareg(
    unsupported_question,
    k=5
)

print("=" * 80)
print("UNSUPPORTED QUESTION")
print("=" * 80)

print(unsupported_question)

print("\nANSWER")
print("=" * 80)

print(answer)

print("\nRETRIEVED SOURCES")
print("=" * 80)

for i, source in enumerate(sources, 1):

    print(f"\nSource {i}")
    print("Document:", source["source"])
    print("Page:", source["page"])
    print("Distance:", round(source["score"], 4))

UNSUPPORTED QUESTION
What is DRAP's regulation for manufacturing medicines on Mars?

ANSWER
Good Distribution Practices for Pharmaceutical and Biological Drug Products (Edition 01) Page 7 of 35 Division of Quality Assurance and Lab Testing, DRAP Act 2012, Drug Act 1976 and rules frame there under

RETRIEVED SOURCES

Source 1
Document: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf
Page: 1
Distance: 0.7441

Source 2
Document: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf
Page: 6
Distance: 0.8161

Source 3
Document: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf
Page: 6
Distance: 0.8461

Source 4
Document: Guidelines-on-Good-Distribution-Practices-for-Pharamceutical-Products.pdf
Page: 5
Distance: 0.8748

Source 5
Document: trs1019-annex5-who-guidelines-on-import-procedures-for-medical-products.pdf
Page: 0
Distance: 0.8788


In [49]:
def check_retrieval_confidence(retrieved_docs, threshold=0.70):

    if not retrieved_docs:
        return False

    best_distance = retrieved_docs[0]["score"]

    return best_distance <= threshold

In [50]:
def ask_pharmareg_safe(question, k=5, threshold=0.70):

    retrieved = retrieve_documents(question, k=k)

    is_relevant = check_retrieval_confidence(
        retrieved,
        threshold=threshold
    )

    if not is_relevant:

        answer = (
            "The available regulatory documents do not "
            "provide sufficient information to answer "
            "this question."
        )

        return answer, retrieved

    prompt = create_rag_prompt(
        question,
        retrieved
    )

    answer = generate_answer(prompt)

    return answer, retrieved

In [51]:
unsupported_question = (
    "What is DRAP's regulation for manufacturing medicines on Mars?"
)

answer, sources = ask_pharmareg_safe(
    unsupported_question,
    k=5,
    threshold=0.70
)

print("=" * 80)
print("QUESTION")
print("=" * 80)

print(unsupported_question)

print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)

print(answer)

print("\n" + "=" * 80)
print("TOP RETRIEVAL DISTANCE")
print("=" * 80)

print(sources[0]["score"])

QUESTION
What is DRAP's regulation for manufacturing medicines on Mars?

ANSWER
The available regulatory documents do not provide sufficient information to answer this question.

TOP RETRIEVAL DISTANCE
0.7440638542175293


In [52]:
threshold = 0.70

threshold_results = []

for item in evaluation_results:

    top_distance = item["sources"][0]["score"]

    threshold_results.append({
        "question": item["question"],
        "distance": top_distance,
        "accepted": top_distance <= threshold
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

                                                                           question  distance  accepted
                                                         What is pharmacovigilance?  0.430765      True
                                     How should biological products be transported?  0.844425     False
                         What are the minimum requirements for hospital pharmacies?  0.584216      True
                                                             What is a drug recall?  0.702091     False
                      What information should be included on pharmaceutical labels?  0.645951      True
                                    What are the requirements for medicine storage?  0.674713      True
               How should temperature-sensitive pharmaceutical products be handled?  0.614655      True
                      What are the responsibilities of pharmaceutical distributors?  0.564630      True
                                                  What is an adv

In [53]:
print("Total valid questions:", len(threshold_df))

print(
    "Accepted by threshold:",
    threshold_df["accepted"].sum()
)

print(
    "Rejected by threshold:",
    (~threshold_df["accepted"]).sum()
)

Total valid questions: 20
Accepted by threshold: 17
Rejected by threshold: 3


In [54]:
rejected_valid = threshold_df[
    threshold_df["accepted"] == False
]

print("Valid questions rejected by threshold:")
print("=" * 80)

for _, row in rejected_valid.iterrows():
    print("\nQuestion:", row["question"])
    print("Distance:", row["distance"])

Valid questions rejected by threshold:

Question: How should biological products be transported?
Distance: 0.8444254398345947

Question: What is a drug recall?
Distance: 0.702090859413147

Question: What are good distribution practices?
Distance: 0.8113980293273926


In [55]:
thresholds = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

print("Threshold Analysis")
print("=" * 60)

for threshold in thresholds:

    accepted = (threshold_df["distance"] <= threshold).sum()
    rejected = (threshold_df["distance"] > threshold).sum()

    print(
        f"Threshold {threshold:.2f} → "
        f"Accepted: {accepted}/20 | "
        f"Rejected: {rejected}/20"
    )

Threshold Analysis
Threshold 0.60 → Accepted: 9/20 | Rejected: 11/20
Threshold 0.65 → Accepted: 13/20 | Rejected: 7/20
Threshold 0.70 → Accepted: 17/20 | Rejected: 3/20
Threshold 0.75 → Accepted: 18/20 | Rejected: 2/20
Threshold 0.80 → Accepted: 18/20 | Rejected: 2/20
Threshold 0.85 → Accepted: 20/20 | Rejected: 0/20
Threshold 0.90 → Accepted: 20/20 | Rejected: 0/20


In [56]:
final_evaluation = []

for i, item in enumerate(evaluation_results, 1):

    top_source = item["sources"][0]

    final_evaluation.append({
        "id": i,
        "question": item["question"],
        "answer": item["answer"],
        "source": top_source["source"],
        "page": top_source["page"],
        "retrieval_distance": top_source["score"]
    })

final_evaluation_df = pd.DataFrame(final_evaluation)

final_evaluation_df

,id,question,answer,source,page,retrieval_distance
0,1,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765
1,2,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425
2,3,What are the minimum requirements for hospital...,"adequate office space, infrastructure, and off...",PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,13,0.584216
3,4,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091
4,5,What information should be included on pharmac...,"name of the product; active ingredient(s), typ...",NPVG-guidelines-reviewed-final-10-10-24-1.pdf,43,0.645951
5,6,What are the requirements for medicine storage?,Temperature and humidity of the medication sto...,p1022drap_1.pdf,26,0.674713
6,7,How should temperature-sensitive pharmaceutica...,Model guidance for the storage and transport o...,trs1019-annex5-who-guidelines-on-import-proced...,11,0.614655
7,8,What are the responsibilities of pharmaceutica...,Distributors/ wholesalers are required to repo...,Guidelines-on-Good-Distribution-Practices-for-...,5,0.564630
8,9,What is an adverse drug reaction?,a response to a drug or therapeutic goods whic...,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,5,0.441929
9,10,What are the requirements for monitoring medic...,Temperature and humidity of the medication sto...,Guidelines-on-Good-Distribution-Practices-for-...,22,0.547578


In [57]:
final_evaluation_df["correct"] = labels_df["correct"].values
final_evaluation_df["complete"] = labels_df["complete"].values
final_evaluation_df["source_relevant"] = labels_df["source_relevant"].values

final_evaluation_df

,id,question,answer,source,page,retrieval_distance,correct,complete,source_relevant
0,1,What is pharmacovigilance?,ensuring patient safety,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,89,0.430765,1,0,1
1,2,How should biological products be transported?,in containers and vehicles,Guidelines-on-Good-Distribution-Practices-for-...,31,0.844425,1,0,1
2,3,What are the minimum requirements for hospital...,"adequate office space, infrastructure, and off...",PHSR-GL-HP-012-Guidelines-on-Standards-for-Est...,13,0.584216,1,0,1
3,4,What is a drug recall?,A process for withdrawing or removing a medica...,Guidelines-on-Good-Distribution-Practices-for-...,28,0.702091,1,1,1
4,5,What information should be included on pharmac...,"name of the product; active ingredient(s), typ...",NPVG-guidelines-reviewed-final-10-10-24-1.pdf,43,0.645951,1,0,1
5,6,What are the requirements for medicine storage?,Temperature and humidity of the medication sto...,p1022drap_1.pdf,26,0.674713,1,0,1
6,7,How should temperature-sensitive pharmaceutica...,Model guidance for the storage and transport o...,trs1019-annex5-who-guidelines-on-import-proced...,11,0.614655,1,0,1
7,8,What are the responsibilities of pharmaceutica...,Distributors/ wholesalers are required to repo...,Guidelines-on-Good-Distribution-Practices-for-...,5,0.564630,1,0,1
8,9,What is an adverse drug reaction?,a response to a drug or therapeutic goods whic...,NPVG-guidelines-reviewed-final-10-10-24-1.pdf,5,0.441929,1,0,1
9,10,What are the requirements for monitoring medic...,Temperature and humidity of the medication sto...,Guidelines-on-Good-Distribution-Practices-for-...,22,0.547578,1,0,1


In [58]:
print("=" * 60)
print("FINAL RAG EVALUATION")
print("=" * 60)

accuracy = final_evaluation_df["correct"].mean()
completeness = final_evaluation_df["complete"].mean()
source_relevance = final_evaluation_df["source_relevant"].mean()

print(f"Answer Correctness:   {accuracy:.2%}")
print(f"Answer Completeness:  {completeness:.2%}")
print(f"Source Relevance:     {source_relevance:.2%}")

FINAL RAG EVALUATION
Answer Correctness:   100.00%
Answer Completeness:  10.00%
Source Relevance:     100.00%


In [59]:
final_evaluation_df.to_csv(
    "pharmareg_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [60]:
import os

print(os.path.exists("pharmareg_evaluation_results.csv"))

True


In [61]:
vectorstore.save_local("pharmareg_faiss")

print("FAISS database saved successfully!")

FAISS database saved successfully!


In [93]:
%%writefile app.py

import streamlit as st
import torch

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="PharmaRegAI",
    page_icon="🧪",
    layout="wide"
)


# =========================================================
# CUSTOM CSS
# =========================================================

st.markdown("""
<style>

/* =========================================================
   GLOBAL
========================================================= */

.stApp {
    background-color: #F5F7F8;
    color: #111827;
}

.main {
    background-color: #F5F7F8;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 3rem;
    max-width: 1200px;
}


/* =========================================================
   HERO TITLE
========================================================= */

.hero-title {
    font-size: 44px;
    font-weight: 800;
    color: #071A21;
    margin-bottom: 6px;
    letter-spacing: -1px;
}

.hero-subtitle {
    font-size: 17px;
    font-weight: 500;
    color: #37474F;
    margin-bottom: 28px;
}


/* =========================================================
   SECTION HEADINGS
========================================================= */

.section-title {
    font-size: 23px;
    font-weight: 750;
    color: #071A21;
    margin-top: 28px;
    margin-bottom: 14px;
}


/* =========================================================
   ANSWER CARD
========================================================= */

.answer-card {
    background-color: #FFFFFF;
    color: #111827;

    padding: 26px;

    border-radius: 14px;

    border: 1px solid #CBD5D9;

    box-shadow: 0 4px 16px rgba(7, 26, 33, 0.08);

    margin-top: 15px;

    font-size: 16px;
    line-height: 1.7;
}


/* =========================================================
   SOURCE CARD
========================================================= */

.source-card {
    background-color: #FFFFFF;
    color: #111827;

    padding: 18px;

    border-radius: 12px;

    border: 1px solid #CBD5D9;

    box-shadow: 0 2px 8px rgba(7, 26, 33, 0.06);

    margin-bottom: 12px;

    font-size: 14px;
    line-height: 1.6;
}

.source-card strong {
    color: #071A21;
}


/* =========================================================
   SIDEBAR
========================================================= */

section[data-testid="stSidebar"] {
    background-color: #071A21;
}

section[data-testid="stSidebar"] * {
    color: #FFFFFF !important;
}

section[data-testid="stSidebar"] h1,
section[data-testid="stSidebar"] h2,
section[data-testid="stSidebar"] h3 {
    color: #FFFFFF !important;
}


/* =========================================================
   TEXT INPUT
========================================================= */

textarea {
    background-color: #FFFFFF !important;
    color: #111827 !important;

    border: 2px solid #B8C4C8 !important;

    border-radius: 10px !important;
}

textarea:focus {
    border-color: #087F8C !important;
    box-shadow: 0 0 0 2px rgba(8, 127, 140, 0.15) !important;
}


/* =========================================================
   BUTTONS
========================================================= */

.stButton > button {
    background-color: #087F8C;
    color: #FFFFFF;

    border: none;

    border-radius: 9px;

    font-weight: 650;

    padding: 10px 18px;

    transition: all 0.2s ease;
}

.stButton > button:hover {
    background-color: #065F69;
    color: #FFFFFF;
    border: none;
}


/* =========================================================
   PRIMARY ASK BUTTON
========================================================= */

.stButton > button[kind="primary"] {
    background-color: #071A21;
    color: #FFFFFF;

    font-weight: 700;

    border-radius: 10px;
}

.stButton > button[kind="primary"]:hover {
    background-color: #087F8C;
    color: #FFFFFF;
}


/* =========================================================
   DISCLAIMER
========================================================= */

.disclaimer {
    background-color: #FFF7D6;

    border: 1px solid #E6C94C;

    padding: 16px;

    border-radius: 10px;

    color: #3D3500;

    font-size: 14px;

    line-height: 1.6;

    margin-top: 30px;
}


/* =========================================================
   FOOTER
========================================================= */

.footer {
    text-align: center;

    color: #4B5A60;

    font-size: 13px;

    margin-top: 40px;

    padding-top: 20px;

    border-top: 1px solid #CBD5D9;
}


/* =========================================================
   DIVIDERS
========================================================= */

hr {
    border-color: #CBD5D9 !important;
}


/* =========================================================
   SPINNER
========================================================= */

.stSpinner > div {
    border-top-color: #087F8C !important;
}

</style>
""", unsafe_allow_html=True)


# =========================================================
# HEADER
# =========================================================

st.markdown(
    '<div class="hero-title">🧪 PharmaRegAI</div>',
    unsafe_allow_html=True
)

st.markdown(
    """
    <div class="hero-subtitle">
    Retrieval-Augmented Regulatory Intelligence System for
    Pharmaceutical Supply Chains
    </div>
    """,
    unsafe_allow_html=True
)


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    st.header("About PharmaRegAI")

    st.write(
        """
        PharmaRegAI retrieves information from pharmaceutical
        regulatory documents and generates answers using
        retrieval-augmented generation (RAG).
        """
    )

    st.divider()

    st.subheader("Technology")

    st.write("• Sentence Transformers")
    st.write("• FAISS Vector Search")
    st.write("• FLAN-T5")
    st.write("• LangChain")
    st.write("• Streamlit")

    st.divider()

    st.subheader("Document Sources")

    st.write("DRAP")
    st.write("WHO")
    st.write("Pharmaceutical Guidelines")

    st.divider()

    st.caption(
        "Developed as a Data Science / AI research project."
    )


# =========================================================
# LOAD EMBEDDING MODEL
# =========================================================

@st.cache_resource
def load_embeddings():

    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )


# =========================================================
# LOAD VECTOR DATABASE
# =========================================================

@st.cache_resource
def load_vectorstore():

    embeddings = load_embeddings()

    db = FAISS.load_local(
        "pharmareg_faiss",
        embeddings,
        allow_dangerous_deserialization=True
    )

    return db


# =========================================================
# LOAD FLAN-T5
# =========================================================

@st.cache_resource
def load_model():

    model_name = "google/flan-t5-base"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name
    )

    return tokenizer, model


# =========================================================
# GENERATE ANSWER
# =========================================================

def generate_answer(prompt):

    tokenizer, model = load_model()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# =========================================================
# CREATE RAG PROMPT
# =========================================================

def create_rag_prompt(question, retrieved_docs):

    context = ""

    for i, doc in enumerate(retrieved_docs, 1):

        context += f"""
SOURCE {i}

Document:
{doc["source"]}

Page:
{doc["page"]}

Content:
{doc["text"]}

-----------------------------------
"""

    prompt = f"""
You are PharmaRegAI, a pharmaceutical regulatory
information assistant.

Answer the question using ONLY the regulatory
documents provided below.

Rules:

1. Do not invent information.
2. Do not use outside knowledge.
3. If the documents do not contain enough information,
   say that the available documents do not provide
   sufficient information.
4. Give a clear and concise answer.
5. Mention the document name and page number when possible.

REGULATORY DOCUMENTS:

{context}

QUESTION:

{question}

ANSWER:
"""

    return prompt


# =========================================================
# RETRIEVAL
# =========================================================

def retrieve_documents(query, k=5):

    vectorstore = load_vectorstore()

    results = vectorstore.similarity_search_with_score(
        query,
        k=k
    )

    retrieved = []

    for doc, score in results:

        retrieved.append({
            "text": doc.page_content,
            "source": doc.metadata.get("source_file"),
            "page": doc.metadata.get("page"),
            "score": float(score)
        })

    return retrieved


# =========================================================
# QUESTION INPUT
# =========================================================

st.markdown(
    '<div class="section-title">Ask a Regulatory Question</div>',
    unsafe_allow_html=True
)

question = st.text_area(
    "Enter your question",
    placeholder="Example: How should temperature-sensitive pharmaceutical products be handled?",
    height=110
)


# =========================================================
# EXAMPLE QUESTIONS
# =========================================================

st.markdown("**Example questions:**")

example_questions = [
    "What is pharmacovigilance?",
    "How should biological products be transported?",
    "What is a drug recall?",
    "What are the requirements for medicine storage?"
]

cols = st.columns(4)

for i, example in enumerate(example_questions):

    with cols[i]:

        if st.button(
            example,
            key=f"example_{i}",
            use_container_width=True
        ):
            question = example


# =========================================================
# ASK BUTTON
# =========================================================

if st.button(
    "🔍 Ask PharmaRegAI",
    type="primary",
    use_container_width=True
):

    if not question.strip():

        st.warning("Please enter a regulatory question.")

    else:

        with st.spinner(
            "Searching regulatory documents and generating answer..."
        ):

            retrieved = retrieve_documents(
                question,
                k=5
            )

            prompt = create_rag_prompt(
                question,
                retrieved
            )

            answer = generate_answer(prompt)


        # =================================================
        # ANSWER
        # =================================================

        st.markdown(
            '<div class="section-title">🤖 Answer</div>',
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="answer-card">
            {answer}
            </div>
            """,
            unsafe_allow_html=True
        )


        # =================================================
        # SOURCES
        # =================================================

        st.markdown(
            '<div class="section-title">📚 Retrieved Sources</div>',
            unsafe_allow_html=True
        )

        for i, doc in enumerate(retrieved[:3], 1):

            st.markdown(
                f"""
                <div class="source-card">

                <strong>Source {i}</strong><br><br>

                📄 <strong>Document:</strong>
                {doc["source"]}<br>

                📑 <strong>Page:</strong>
                {doc["page"]}<br>

                🔎 <strong>Retrieval distance:</strong>
                {doc["score"]:.4f}

                </div>
                """,
                unsafe_allow_html=True
            )


# =========================================================
# DISCLAIMER
# =========================================================

st.markdown(
    """
    <div class="disclaimer">

    ⚠️ <strong>Important:</strong>
    PharmaRegAI is an academic/research prototype.
    It provides information retrieved from the included
    regulatory documents and should not replace official
    regulatory guidance or professional pharmaceutical advice.

    </div>
    """,
    unsafe_allow_html=True
)


# =========================================================
# FOOTER
# =========================================================

st.markdown(
    """
    <div class="footer">
    PharmaRegAI • Retrieval-Augmented Regulatory Intelligence
    </div>
    """,
    unsafe_allow_html=True
)

Overwriting app.py


In [64]:
import os

print(os.path.exists("app.py"))

True


In [65]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 17.2 MB/s eta 0:00:00


In [66]:
!streamlit run app.py &>/content/streamlit.log &

In [67]:
!cat /content/streamlit.log



2026-09-26 06:30:41.324 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.81.148:8501



In [69]:
!pip install -q pyngrok

In [72]:
from pyngrok import ngrok
from google.colab import userdata

# Get your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# and add it to Colab Secrets named 'NGROK_AUTH_TOKEN'
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8501)

print("PharmaRegAI Demo URL:")
print(public_url)

PharmaRegAI Demo URL:
NgrokTunnel: "https://lively-zone-mouth.ngrok-free.dev" -> "http://localhost:8501"


In [73]:
!pip install -q langchain-huggingface

In [75]:
!pkill -f streamlit

In [76]:
!streamlit run app.py &>/content/streamlit.log &

In [78]:
!pip install -q langchain-community langchain-huggingface

In [79]:
with open("app.py", "r") as f:
    app_code = f.read()

app_code = app_code.replace(
    "from langchain.embeddings import HuggingFaceEmbeddings",
    "from langchain_huggingface import HuggingFaceEmbeddings"
)

app_code = app_code.replace(
    "from langchain.vectorstores import FAISS",
    "from langchain_community.vectorstores import FAISS"
)

with open("app.py", "w") as f:
    f.write(app_code)

print("All LangChain imports fixed!")

All LangChain imports fixed!


In [80]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ HuggingFaceEmbeddings import works")
print("✅ FAISS import works")

✅ HuggingFaceEmbeddings import works
✅ FAISS import works


In [77]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("PharmaRegAI Demo URL:")
print(public_url)

PharmaRegAI Demo URL:
NgrokTunnel: "https://lively-zone-mouth.ngrok-free.dev" -> "http://localhost:8501"


In [94]:
!pkill -f streamlit

In [95]:
!streamlit run app.py &>/content/streamlit.log &

In [96]:
!cat /content/streamlit.log

In [97]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("PharmaRegAI Demo URL:")
print(public_url)

PharmaRegAI Demo URL:
NgrokTunnel: "https://lively-zone-mouth.ngrok-free.dev" -> "http://localhost:8501"


In [90]:
%%writefile requirements.txt

streamlit
langchain
langchain-community
langchain-huggingface
sentence-transformers
faiss-cpu
pypdf
transformers
torch
accelerate

Writing requirements.txt


In [91]:
%%writefile README.md

# PharmaRegAI
### Retrieval-Augmented Regulatory Intelligence System for Pharmaceutical Supply Chains

PharmaRegAI is an AI-powered regulatory information assistant designed to retrieve relevant information from pharmaceutical regulatory documents and generate concise, source-grounded answers.

The system uses Retrieval-Augmented Generation (RAG) to reduce unsupported responses by retrieving relevant regulatory passages before generating an answer.

---

## Project Overview

Pharmaceutical supply chains operate under extensive regulatory requirements involving:

- Medicine storage
- Pharmaceutical transportation
- Good Distribution Practices
- Pharmacovigilance
- Drug recalls
- Hospital pharmacy standards
- Temperature-sensitive products
- Pharmaceutical records
- Product quality

Finding relevant information across large regulatory documents can be time-consuming.

PharmaRegAI addresses this problem by combining semantic search with a lightweight language model.

The system retrieves relevant sections from regulatory documents and provides an answer together with the source document and page number.

---

## System Architecture

```text
Regulatory PDF Documents
          |
          v
     PDF Extraction
          |
          v
      Text Chunking
          |
          v
 Sentence Transformer
    Embeddings
          |
          v
      FAISS Vector
        Database
          |
          v
   Semantic Retrieval
          |
          v
       FLAN-T5
     Answer Generation
          |
          v
 Answer + Source Pages
          |
          v
       Streamlit
       Web Interface

Writing README.md


In [92]:
import os

files_to_check = [
    "app.py",
    "README.md",
    "requirements.txt",
    "pharmareg_evaluation_results.csv",
    "pharmareg_faiss/index.faiss",
    "pharmareg_faiss/index.pkl"
]

for file in files_to_check:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌ Missing:", file)

✅ app.py
✅ README.md
✅ requirements.txt
✅ pharmareg_evaluation_results.csv
✅ pharmareg_faiss/index.faiss
✅ pharmareg_faiss/index.pkl
